In [ ]:
import numpy as np
import pandas as pd
import csv
from sklearn.model_selection import train_test_split
from datasets import DatasetDict, Dataset


In [ ]:
train_df= pd.read_csv('/content/drive/MyDrive/w266-Project/TextGenerate Summary/ESM_Train_filteredDescription.csv')
eval_df = pd.read_csv('/content/drive/MyDrive/w266-Project/TextGenerate Summary/ESM_eval_filteredDescription.csv')
test_df = pd.read_csv('/content/drive/MyDrive/w266-Project/TextGenerate Summary/ESM_eval_filteredDescription.csv')


dataset_main = pd.concat([train_df,eval_df,test_df], ignore_index = True)


In [ ]:
dataset_main.head()


,key,concatenated_description,filtered_description
0,0,Catalyzes the facilitated diffusion of 2-acyl-...,catalyzes facilitated diffusion cell participa...
1,1,Reversibly catalyzes the transfer of the carba...,reversibly catalyzes transfer carbamoyl group ...
2,2,Involved in pH regulation to eliminate acids g...,involved ph regulation eliminate acids generat...
3,3,Thiol protease that seems to be involved in th...,thiol protease seems involved sexual developme...
4,4,Catalyzes the attachment of tryptophan to tRNA...,catalyzes attachment tryptophan trna trp part ...


In [ ]:
import nltk
import string
from rake_nltk import Rake
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')
rake_nltk_var = Rake()

def remove_punctuation(text):
    # Tokenize the text
    words = word_tokenize(text)
    # Filter out punctuation
    words = [word for word in words if word.isalnum()]
    # Join the words back into a single string
    return ' '.join(words)

def extract_keywords(text):
    rake_nltk_var.extract_keywords_from_text(text)
    keyword_extracted = rake_nltk_var.get_ranked_phrases()[:5]
    return ' '.join(keyword_extracted)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
dataset_main['filtered_description'] = dataset_main['filtered_description'].apply(remove_punctuation)



In [ ]:
dataset_main['filtered_description'] = dataset_main['filtered_description'].apply(extract_keywords)

In [ ]:
dataset_main.head()

,key,concatenated_description,filtered_description
0,0,Catalyzes the facilitated diffusion of 2-acyl-...,catalyzes facilitated diffusion cell participa...
1,1,Reversibly catalyzes the transfer of the carba...,reversibly catalyzes transfer carbamoyl group ...
2,2,Involved in pH regulation to eliminate acids g...,involved ph regulation eliminate acids generat...
3,3,Thiol protease that seems to be involved in th...,thiol protease seems involved sexual developme...
4,4,Catalyzes the attachment of tryptophan to tRNA...,catalyzes attachment tryptophan trna trp part ...


In [ ]:
dataset_main = train_df.rename(columns={
    'filtered_description': 'text',
    'concatenated_description': 'summary'
})

In [ ]:
text_pairs = dataset_main.apply(lambda row: {'text': 'summarize: ' + row['text'], 'summary': row['summary']}, axis=1).tolist()

In [ ]:
for _ in range(5):
    print(np.random.choice(text_pairs))

{'text': 'summarize: produces atp adp presence proton gradient across membrane gamma chain believed important regulating atpase activity flow protons cf 0 complex catalyzes transfer enolpyruvyl moiety phosphoenolpyruvate pep s3p produce enolpyruvyl inorganic phosphate catalyzes phosphorylation fructose atp first committing step glycolysis', 'summary': 'Produces ATP from ADP in the presence of a proton gradient across the membrane. The gamma chain is believed to be important in regulating ATPase activity and the flow of protons through the CF(0) complex. Catalyzes the transfer of the enolpyruvyl moiety of phosphoenolpyruvate (PEP) to the 5-hydroxyl of shikimate-3-phosphate (S3P) to produce enolpyruvyl shikimate-3-phosphate and inorganic phosphate. Catalyzes the phosphorylation of D-fructose 6-phosphate to fructose 1,6-bisphosphate by ATP, the first committing step of glycolysis.'}
{'text': 'summarize: together lpte involved assembly lipopolysaccharide lps surface outer membrane mediates

In [ ]:
len(text_pairs)

48251

In [ ]:
np.random.shuffle(text_pairs)

In [ ]:
train_data, temp_data = train_test_split(text_pairs, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5,random_state = 42 )

print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")
print(f"Test set size: {len(test_data)}")

Training set size: 38600
Validation set size: 4825
Test set size: 4826


In [ ]:
train_file = '/content/drive/MyDrive/w266-Project/T5Summarization/train_pairs.csv'
valid_file = '/content/drive/MyDrive/w266-Project/T5Summarization/valid_pairs.csv'
test_file = '/content/drive/MyDrive/w266-Project/T5Summarization/test_pairs.csv'

pd.DataFrame(train_data).to_csv(train_file)
pd.DataFrame(val_data).to_csv(valid_file)
pd.DataFrame(test_data).to_csv(test_file)

In [ ]:
train_data = pd.read_csv('/content/drive/MyDrive/w266-Project/T5Summarization/train_t5.csv')
val_data = pd.read_csv('/content/drive/MyDrive/w266-Project/T5Summarization/valid_t5.csv')
test_data = pd.read_csv('/content/drive/MyDrive/w266-Project/T5Summarization/test_t5.csv')

In [ ]:
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

dataset_dict = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})



In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
def preprocess_function(examples):
    inputs = examples["text"]

    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)

    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
tokenized_dataset_dict = dataset_dict.map(preprocess_function, batched=True)


Map:   0%|          | 0/38600 [00:00<?, ? examples/s]

Map:   0%|          | 0/4825 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model="t5-small")

In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

Experimentation of Freezing Encoder (Decrease Parameter in hopes of both decreasing computational load and increasing summarization ability.)

In [ ]:
#Freezing the first half of the encoder
num_encoder_layers = len(model.encoder.block)
half_encoder_layers = num_encoder_layers // 2

for i in range(half_encoder_layers):
    for param in model.encoder.block[i].parameters():
        param.requires_grad = False

for i in range(half_encoder_layers):
    for param in model.encoder.block[i].parameters():
        assert param.requires_grad == False


for i in range(half_encoder_layers, num_encoder_layers):
    for param in model.encoder.block[i].parameters():
        assert param.requires_grad == True

for param in model.decoder.parameters():
  assert param.requires_grad == True

In [ ]:
#Freezing the entire encoder.
for param in model.encoder.parameters():
    param.requires_grad = False

#Checking if encoder is frozen
for param in model.encoder.parameters():
    assert param.requires_grad == False


for param in model.decoder.parameters():
    param.requires_grad = True
#Checking if Decoder is not Frozen
for param in model.decoder.parameters():
    assert param.requires_grad == True

In [ ]:
import torch

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/fine_tuned_t5_small_HFE_model",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=4,
    predict_with_generate=True,
    fp16=True,
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
total_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(total_trainable_params)

41625344


In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset_dict["train"],
    eval_dataset=tokenized_dataset_dict["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("/content/drive/MyDrive/w266-Project/T5Summarization/ProtCosineSim_T5_small_HFE_model/ProtCosine_fine_tuned_t5_small_HFE_model")

INFERENCE

In [ ]:
import pandas as pd
from datasets import DatasetDict, Dataset
import torch
from torch.utils.data import DataLoader

In [ ]:
test_df = pd.read_csv('/content/drive/MyDrive/w266-Project/T5Summarization/test_t5.csv')

In [ ]:
test_df.head()

,Unnamed: 0,text,summary
0,0,summarize: rna helicase may involved spliceoso...,ATP-dependent RNA helicase which may be involv...
1,1,summarize: inhibitor lipoprotein binding low d...,Inhibitor of lipoprotein binding to the low de...
2,2,summarize: synthetase relaxed trna specificity...,Aspartyl-tRNA synthetase with relaxed tRNA spe...
3,3,summarize: purine salvage pathway enzyme catal...,Purine salvage pathway enzyme that catalyzes t...
4,4,summarize: catalyzes conversion catalyzes atta...,Catalyzes the conversion of glucosamine-6-phos...


In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.texts = texts
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        tokenized_text = self.tokenizer(
            text,
            return_tensors='pt'
        )
        return {
            'input_ids': tokenized_text['input_ids'].squeeze(),
            'attention_mask': tokenized_text['attention_mask'].squeeze()
        }

In [ ]:
total_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(total_trainable_params)

60506624


In [ ]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained('/content/drive/MyDrive/w266-Project/T5Summarization/ProtCosineSim_T5_small_FE_model/ProtCosine_fine_tuned_t5_small_FE_model')
model = AutoModelForSeq2SeqLM.from_pretrained('/content/drive/MyDrive/w266-Project/T5Summarization/ProtCosineSim_T5_small_FE_model/ProtCosine_fine_tuned_t5_small_FE_model')
model.eval()
model.to('cuda')
#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
import time
from tqdm import tqdm

def generate_summary(input_texts, max_length= 1024):
    inputs = tokenizer(input_texts,padding='max_length', truncation=True, max_length=max_length, return_tensors='pt').to('cuda')
    input_ids = inputs['input_ids'].to('cuda')
    attention_mask = inputs['attention_mask'].to('cuda')

    with torch.no_grad():
      outputs = model.generate(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'], max_new_tokens=200, do_sample=False)
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

def batch_data(data, batch_size):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

def process_batches(data, batch_size):
    summaries = []
    start_time = time.time()

    # Determine the total number of batches
    num_batches = len(data) // batch_size + (1 if len(data) % batch_size != 0 else 0)

    # Create the progress bar
    with tqdm(total=num_batches, desc="Processing Batches") as pbar:
        for batch in batch_data(data, batch_size):
            batch_summaries = generate_summary(batch)
            summaries.extend(batch_summaries)

            torch.cuda.empty_cache()

            pbar.update(1)

    end_time = time.time()

    total_time = end_time - start_time
    average_batch_time = total_time / num_batches

    print(f"Total inference time: {total_time:.2f} seconds")
    print(f"Average inference time per batch: {average_batch_time:.2f} seconds")

    return summaries



test_data = test_df['text'].tolist()
batch_size = 8
summaries = process_batches(test_data, batch_size)



Processing Batches: 100%|██████████| 604/604 [16:27<00:00,  1.63s/it]

Total inference time: 987.23 seconds
Average inference time per batch: 1.63 seconds


In [ ]:
test_df['generated_summary'] = summaries
test_df.to_csv('/content/drive/MyDrive/w266-Project/T5Summarization/Output/t5small_FE_output.csv', index=False)

In [ ]:
print(len(test_df['summary'][0]))
print(len(test_df['generated_summary'][0]))

428
360


In [ ]:
from datasets import load_metric

rouge = load_metric('rouge')

<ipython-input-18-d4c0aae63b14>:3: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  rouge = load_metric('rouge')


The repository for rouge contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/rouge.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


In [ ]:
results = rouge.compute(predictions=summaries, references=test_df['summary'].tolist(), use_stemmer=True)


In [ ]:
print("Average ROUGE scores:")
print(f"ROUGE-1: {results['rouge1'].mid.fmeasure:.4f}")
print(f"ROUGE-2: {results['rouge2'].mid.fmeasure:.4f}")
print(f"ROUGE-L: {results['rougeL'].mid.fmeasure:.4f}")

Average ROUGE scores:
ROUGE-1: 0.7184
ROUGE-2: 0.5120
ROUGE-L: 0.6730
